# Sales Forecasting with PyCaret - Exogenous Variables
# Part A: Exploratory Data Analysis
## Dr José M Albornoz
### June 2025

In this notebook we focus on the exploratory data analysys of a dataset that describes retail sales at a department store; exogenous variables that the SMEs consider may drive sales are considered. PyCaret will be used throughout this notebook.

# 0.- Imports

In [1]:
from pycaret.time_series import *
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objs as go
from plotly.offline import iplot
from plotly.subplots import make_subplots
import datetime
import seaborn as sns
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import ccf # For calculating Cross-Correlation Function
from statsmodels.graphics.tsaplots import plot_acf # We can adapt plot_acf for cross-correlation plotting

# maximum number of rdataframe ows and columns displayed
pd.set_option('display.max_rows', 50000)
pd.set_option('display.max_columns', 500)

pd.options.mode.chained_assignment = None

RANDOM_SEED = 801

# 1.- Load data

In [2]:
data = pd.read_csv('../data/sales_with_exogenous.csv')
target = "Sales"
data.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/sales_with_exogenous.csv'

In [ ]:
data.shape

In [ ]:
# feature types
data.dtypes

We find that there are 3 text features: **Marketing**, **Holiday** and **DestinationEvent**.

In [ ]:
# number of missing values
data.isnull().sum()

In [ ]:
# number of unique values
data.nunique()

We can see that the macroeconomic indicators in the dataset have missing values; we will use forward imputation to remove those values for the purposes of our exploratory data analysis

In [ ]:
data = data.ffill()

# 2.- Feature engineering for text features

We have several text features in our dataset. PyCaret, like most machine learning frameworks, requires all features to be numerical. Therefore, these text features must first be transformed into numerical features. We will first deal with the simple case of features representing binary choices: **Holiday** and **DestinationEvent**.

In [ ]:
data_engineered = data.copy()

In [ ]:
data_engineered['Holiday'] = data_engineered['Holiday'].map({'Yes': 1, 'No': 0})

In [ ]:
data_engineered['DestinationEvent'] = data_engineered['DestinationEvent'].map({'Yes': 1, 'No': 0})

In [ ]:
data_engineered.head()

In [ ]:
# save dataframe with encoded Holiday and DestinationEvent
data_engineered.to_csv('../data/sales_with_exogenous_partial_text_encoding.csv', index=False)

In [ ]:
# convert Date to datetime
data_engineered['Date'] = pd.to_datetime(data_engineered['Date'])

# set Date as index
data_engineered.set_index('Date', inplace=True)

**Marketing** contains text describing marketing events happening at the store on a given day, such as *August In Store Credit Card Signup Discount; Back to School Mailer and Billboard Campaign ID4.2822*. 

A NLP approach such as Bag of Words or TF-IDF could be used to transform this feature; however these techniques can lead to very high dimensionality, as columns will be generated for each unique word. For this reason we are not going to consider **Marketing** for EDA purposes. 

In [ ]:
# drop Marketing from the dataset
data_engineered.drop(columns='Marketing', inplace=True)

# 2.- PyCaret Setup

The `setup` function initializes the training environment and creates the transformation pipeline. We will take advantage of this function for our EDA.

In [ ]:
data_engineered.head()

In [ ]:
# We want to forecast the next 14 days, and we will use 3 fold cross-validation to test the models.
fh = 14 # or alternately fh = np.arange(1,13)
fold = 3 # (default)

As it can be observed, there are missing values corresponding to the macroeconomic indicators (EconChangeGDP, EconJobsChange and Annualized CPI) as they have quarterly, monthly and weekly frequencies. To deal with this issue we are going to perform forward fill imputation on these features through `setup`.

In [ ]:
s = setup(data_engineered, 
          target = 'Sales', 
          fh = fh, 
          fold = fold, 
          numeric_imputation_exogenous="ffill",
          session_id = RANDOM_SEED)

# 3.- Exploratory data analysis

We already performed EDA for the target **Sales**, so here we will focus on finding possible connections between **Sales** and the exogenous variables included in the dataset. 

## 3.1.- Correlation between target and exogenous variables

We would like to examine the correlations between our target and each of the exogenous variables. We will do that in the next code cells:

In [ ]:
# access the preprocessed data (training set)
X_train_processed = get_config('X_train')
y_train_processed = get_config('y_train')

In [ ]:
# the exogenous features are already included in X_train_processed.
# for correlation, we need the target series to be aligned with X_train_processed.
# y_train_processed is already aligned.

# combine processed target and exogenous features into a single DataFrame
combined_df = pd.concat([y_train_processed, X_train_processed], axis=1)

In [ ]:
# calculate the correlation matrix
correlation_matrix = combined_df.corr()

In [ ]:
# visualize the correlation matrix (Recommended)
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title(f'Correlation Matrix: {target} and Exogenous Variables')
plt.show()

We observe that the correlation between the target and the exogenous variables exceeds 0.5 for **Num_Employees**, **Pct_On_Sale**, and **Holiday**. There are 4 exogenous features for which the correlation with sales is negligible: **Pct_Promotional**, **Econ_ChangeGDP**, **EconJobsChange** and **AnnualizedCPI**.

## 3.2.- Crosscorrelation between target and exogenous variables

We will now examine cross correlations to identify lag relationships between exogenous variables and the target time series. 

In [ ]:
# exogenous variables in dataset 
exog_cols = ['Num_Employees', 'Returns_Pct', 'Pct_On_Sale', 'Pct_Promotional', 'Holiday', 'DestinationEvent', 'Econ_ChangeGDP', 
            'EconJobsChange', 'AnnualizedCPI']

In [ ]:
# calculate and plot cross-correlations for each exogenous variable
max_lags = 7 # to be expected in physical stores

for exog_col in exog_cols:
    # calculate cross-correlation function
    # the ccf function expects numpy arrays
    ccf_values = ccf(y_train_processed.values, X_train_processed[exog_col].values, adjusted=False)

    # plotting 
    plt.figure(figsize=(10, 4))

    # only positive lags for clearer interpretation of exog's influence on target
    lags = np.arange(0, max_lags + 1)
    plt.stem(lags, ccf_values[:max_lags+1], basefmt=" ", markerfmt="o", linefmt="-")
    plt.axhline(0, color='gray', linestyle='--')
    # Add significance lines (approximate for large N, typically 2/sqrt(N))
    conf_level = 1.96 / np.sqrt(len(y_train_processed))
    plt.axhspan(-conf_level, conf_level, color='blue', alpha=0.1) # Approx. 95% CI
    plt.title(f'Cross-Correlation Function (CCF) between {target} and {exog_col}')
    plt.xlabel('Lag')
    plt.ylabel('Cross-Correlation')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

We can see that there are exogenous variables with negligible crosscorrelation values across all considered lags: **Pct_Promotional**, **Econ_ChangeGDP**, **Econ_JobsChange**, and **AnnualizedCPI**. We won't perform any further analysis for these features.

## 3.3.- Exploratory plots

We will now plot **Sales** against each of the exogenous variables to visually explore their association>

In [ ]:
# plot target along with exogenous variables 
# plotting an interactive plot can slow down the notebook - hence, we will revert to a static renderer for this plot
s.plot_model(fig_kwargs={"renderer": "png", "width": 1000, "height": 1200})

The above plot clearly shows that there are features such as **Returns_Pct** and **Pct_On_Sale** that appear to be associated with periods of increased sales (i.e. Christmas).

We will start by plotting **Num_Employees** vs. **Sales**.

In [ ]:
trace1 = go.Bar(
    x = data_engineered.index,
    y = data_engineered['Sales'],
    name = 'Sales',
    marker=dict(color='rgb(34,60,192)')
)
trace2 = go.Scatter(
    x = data_engineered.index,
    y = data_engineered['Num_Employees']*6000,
    name='Number of Employees',
)

fig = make_subplots(rows=1, cols=1)
fig.add_trace(trace1, row=1, col=1)
fig.add_trace(trace2, row=1, col=1)
fig['layout'].update(height = 600, width = 1100, title = 'Sales vs Number of Employees', xaxis=dict(tickangle=-90))

iplot(fig)

We now plot Sales vs percentage of items in sale

In [ ]:
trace1 = go.Bar(
    x = data_engineered.index,
    y = data_engineered['Sales'],
    name = 'Sales',
    marker=dict(color='rgb(34,60,192)')
)
trace2 = go.Scatter(
    x = data_engineered.index,
    y = data_engineered['Pct_On_Sale']*6000,
    name='Percentage of items on sale',
)

fig = make_subplots(rows=1, cols=1)
fig.add_trace(trace1, row=1, col=1)
fig.add_trace(trace2, row=1, col=1)
fig['layout'].update(height = 600, width = 1100, title = 'Sales vs Percentage of Items on Sale', xaxis=dict(tickangle=-90))

iplot(fig)

The above plot shows that some spikes in the number of employees appear to coincide with spikes in sales. Let's take a closer look at the relation between **Sales** and **Returns_Pct**:

In [ ]:
trace1 = go.Bar(
    x = data_engineered.index,
    y = data_engineered['Sales'],
    name = 'Sales',
    marker=dict(color='rgb(34,60,192)')
)
trace2 = go.Scatter(
    x = data_engineered.index,
    y = data_engineered['Returns_Pct']*5e5,
    name='Item Return Rate',
)

fig = make_subplots(rows=1, cols=1)
fig.add_trace(trace1, row=1, col=1)
fig.add_trace(trace2, row=1, col=1)
fig['layout'].update(height = 800, width = 1100, title = 'Sales vs Item Return Rate', xaxis=dict(tickangle=-90))

iplot(fig)

The increase in sales during the Christmas season seems to be associated with an increase in **Returns_Pct**; also, there are some spikes in sales that coincide with spikes in the returns rate. 

The next two plots aim at examining possible relations between **Holiday** and **DestinationEvent** with sales.

In [ ]:
trace1 = go.Bar(
    x = data_engineered.index,
    y = data_engineered['Sales'],
    name = 'Sales',
    marker=dict(color='rgb(34,60,192)')
)
trace2 = go.Scatter(
    x = data_engineered.index,
    y = data_engineered['Holiday']*800000,
    name='Holidays',
)

fig = make_subplots(rows=1, cols=1)
fig.add_trace(trace1, row=1, col=1)
fig.add_trace(trace2, row=1, col=1)
fig['layout'].update(height = 600, width = 1100, title = 'Sales vs Holidays', xaxis=dict(tickangle=-90))

iplot(fig)

In [ ]:
trace1 = go.Bar(
    x = data_engineered.index,
    y = data_engineered['Sales'],
    name = 'Sales',
    marker=dict(color='rgb(34,60,192)')
)
trace2 = go.Scatter(
    x = data_engineered.index,
    y = data_engineered['DestinationEvent']*800000,
    name='Destination Event',
)

fig = make_subplots(rows=1, cols=1)
fig.add_trace(trace1, row=1, col=1)
fig.add_trace(trace2, row=1, col=1)
fig['layout'].update(height = 600, width = 1100, title = 'Sales vs Destination Events', xaxis=dict(tickangle=-90))

iplot(fig)

We can appreciate that both holidays and destination events are associated with spikes in sales. Our last plot will be **Sales** vs **Eco_ChangeGDP**.

At this point we are satisfied that there are exogenous variables that have a clear influence on our target variable - and therefore it's worthwhile to include them in our forecasting model.